In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Session 6: Policy gradient II: advantage, baseline and REINFORCE

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position**
    - session 6: last session derived the Policy Gradient Theorem
    - this session makes it practical: baselines and advantages, then REINFORCE and the surrogate loss
    - the next session adds PPO and GRPO
- **This session's goal**
    - see how the noisy raw-return weight becomes a stable update
- **By the end you can**
    - say why subtracting a baseline helps and does not bias the gradient
    - name the advantage used by REINFORCE
    - say what the surrogate loss is and why it keeps each update near the old policy


# Policy gradient based methods: concepts

**Why this session?** The theorem weights every action's gradient by the whole episode's return, which makes the update noisy. This session makes it practical.

Recall the Policy Gradient Theorem
- formulated with the episode return $\rlret_{0,\tau}$

$$
\nabla_\theta J(\theta) = 
\Exp{\tau \sim \pi_\theta} { 
\sum_{\tt=0}^{|\tau|-1} {
\nabla_\theta   \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) \, \,\rlret_{0,\tau}
}
} 
$$

- formulated with intermediate rewards

$$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) \, \disc^\tt \rlret_{\tau,\tt} \right]
$$

Policy based methods
update parameters $\theta$
- in the direction (gradient)
- that increases expected return 

# Advantage: Beyond Rewards

**Why this section?** How can we change the weights, to reduce the noise, without changing the expected gradient?

The Policy Gradient Theorem states that the gradient for the trajectory $\tau$ is a sum of per-step gradients
- each weighted by the episode return


But perhaps the policy should be 
- *reinforced (positively or negatively)*
- in different proportions ?


Suppose all returns are positive.
- the Policy Gradient Theorem would push up *every* sampled action
- in proportion to the raw return, even when most of that return would have been obtained anyway

Or suppose we always get "easy" questions correct and only sometimes get "hard" questions correct
- the Policy Gradient Theorem would positively reinforce the already correct easy questions
- with as much force as it positively reinforces the hard question
    - for methods that involve multiple trajectories per example (e.g., GRPO)

The *expected* gradient is the same either way: the problem is **variance** (noise), not bias.


We need some way to *relativize* 
 the return for a state
- by comparing it to a *baseline* (the anticipated return) for the state.
    - often $\statevalfun_\pi ( \stateseq_{\tt})$, the value of the state as would be computed by a Value function
- we call this relativized value the *advantage*

We will use the advantage as a mechanism to differentially weight the gradient of individual actions.





For example: consider a baseline that is the average return from the state
- above average returns have positive advantage
- below average returns have a negative advantage

Thus, in the case of all returns being positive
- we favor actions that result in "above expected" return (positive advantages)
- we disfavor actions that result "below expected" return (negative advantages)

Subtracting a baseline from the return has other advantages
- reduces the *magnitude* of the parameter update
    - smoother training
- reduces "noise"
    - the baseline value for a state is common to *all actions* from the state
    - so the advantage is relative to the signal *only* from the action itself


| Aspect                    | Effect of Subtracting Baseline                         |
|:---------------------------|:-------------------------------------------------------|
| Noise cancellation        | Removes expected (common) return, reducing fluctuations |
| Bias introduction         | None, if the baseline does not depend on the action   |
| Zero-centering            | Advantages centered around zero, stabilizing updates  |



**Why a baseline does not bias the gradient**

A baseline $b(\state)$ may depend on the state but *not on the action*. Then
$$
\E_{\actseq \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \, b(\state) \bigr] = b(\state) \, \E\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \bigr] = 0
$$

- the score function has zero mean ($\sum_\act \pi_\theta \nabla_\theta \log \pi_\theta = \nabla_\theta \sum_\act \pi_\theta = 0$), so subtracting $b$ leaves the *expected* gradient unchanged
- what changes is the variance: a good baseline cancels the part of the return that every action shares
- the **advantage** is the return relative to such a baseline: $\advseq = \actvalfun_\pi(\state, \act) - \statevalfun_\pi(\state)$ when exact, $\rlret_\tt - b(\stateseq_\tt)$ when sampled
- because $\E[\rlret_\tt \mid \state, \act] = \actvalfun_\pi(\state, \act)$, the sampled $\rlret_\tt - b$ is a sample of the advantage; a baseline computed from the same sample (such as a group mean that includes it) is only approximately independent of the action
- *variance*: how much an estimate fluctuates from sample to sample; *bias*: how far its average is from the truth (the self-study appendix compares the estimators)


## Credit Assignment: The Evolution of the Advantage

**Why this section?** One weight for a whole episode cannot say which step deserves the credit. That leads to a weight per step: the advantage.

"Early" Policy based methods (e.g., REINFORCE) were a direct expression of the Policy Gradient Theorem


$$
\nabla_\theta J(\theta) = 
\Exp{\tau \sim \pi_\theta} { 
\sum_{\tt=0}^{|\tau|-1} {
\nabla_\theta   \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) \, \,\rlret_{0,\tau}
}
} 
$$

where the gradient of each step $\tt$ was equally weighted by the episode return ($\rlret_{0,\tau}$).



The problem with this
- some steps are "good": would have led to high episode return
- some steps are "bad":  decreased the potential episode return

But, with sparse rewards-
- we don't know how to *assign credit* for individual steps

For example:

- If an LLM writes a 100-word masterpiece 
- but makes a typo on the very last character
- the final reward might be low. 

Equal weighting of gradients punishes *all* steps whereas a single step is clearly to blame.

By obfuscating good signals from bad signals, learning is slowed
- high variance

Computing an Advantage per step
- is a means of solving the *Credit Assignment* problem
- creating differential degrees of reinforcement

"Modern RL" is associated with a move from Reward toward Advantage (advantage-based methods are older)
- popularised by [Asynchronous Advantage Actor-Critic](https://arxiv.org/abs/1602.01783) 2016

Having a separate Advantage $\advseq_{\tau,\tt}$ for each step $\tt$
- dictates the "Force" and "Direction" of the gradient weight update

| Advantage Value | Effect on Gradient | The "Steering" Result |
| :--- | :--- | :--- |
| **Positive ($\advseq_{\tau,\tt} > 0$)** | Multiplies by a positive scalar. | **Accelerate:** This action was better than the "usual." Make it more likely. |
| **Negative ($\advseq_{\tau,\tt} < 0$)** | Flips the sign of the gradient. | **Brake:** This action fell below expectations. Make it less likely. |
| **Zero ($\advseq_{\tau,\tt} \approx 0$)** | Nullifies the gradient. | **Maintain:** You did exactly what was expected. No learning required. |

Good steps are encouraged; bad (e.g., below expectation) steps are discouraged

The Policy Based methods we will study take different approaches to *Credit Assignment* (computing Advantage).


- REINFORCE (episode-return form)
    - equal credit to every step (the return-to-go form weights steps by the rewards that follow)
- PPO (Surgical Credit)
    - assigns credit to each step
    - by maintaining a prediction of the value of each state (a Value function)
- GRPO (Statistical Credit)
    - Uses the **group mean** of the returns of multiple completions as the baseline, and divides by their **standard deviation** 
    - It gives the **same Advantage** to every token in a specific trajectory: it gives up per-step credit assignment for simplicity; the group only supplies the baseline and the scale (only PPO's critic assigns credit per step)
    - Each completion is one episode $\tau_i$, $i = 1, \ldots, G$: its single reward (written $r_i$ in the papers) is the episode return $\rlret_{0,\tau_i}$, the same quantity as in the *Episode Return form* of the theorem
    - The advantage of completion $i$ is $\left( \rlret_{0,\tau_i} - \operatorname{mean}(\rlret_{0,\tau_1}, \ldots, \rlret_{0,\tau_G}) \right) / \operatorname{std}(\rlret_{0,\tau_1}, \ldots, \rlret_{0,\tau_G})$, written with $\operatorname{mean}$ and $\operatorname{std}$ as in the papers
    - **Notation note:** the GRPO papers (DeepSeekMath, DeepSeek-R1, DAPO) write the *group size* as $G$. The return in these notebooks, $\rlret_\tt$ or $\rlret_{0,\tau}$, is also written $G$ in the literature. We keep both conventions: the group size is a bare count ($i = 1, \ldots, G$) while the return always carries a subscript, so context tells them apart


---

## Unified Policy Gradient Formulation: Advantage Definitions

**Why this section?** The methods differ only in how they compute the advantage. One formula lets us compare them.

Even though different Policy Based methods compute Advantage differently
- we can uniformly implement weighted Policy Gradient updates
- by rewriting the Policy Gradient Theorem in terms of Advantage

$$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) \advseq_{\tau,\tt} \right]
$$

**Note:** as in most implementations, the factor $\disc^\tt$ of the exact discounted gradient is usually dropped (it is $1$ when $\disc = 1$); the formulation is then the common practical form, not exactly the discounted gradient.

This allows us to focus on the *Advantage definition* 
$$
\advseq_{\tau,\tt} 
$$
as a way of differentiating methods.

We thus refer to the above formulation as the Unified Policy Gradient Formulation.

The policy-based methods we present will typically be variations of this "vanilla" gradient.

# Policy gradient based methods: Preview

We will show several common policy-based methods.
- REINFORCE
- PPO (the "classic" method)
- GRPO (relatively new: 2024)

Details of each follow.

As a preview, we show the definition of the Advantage for each.


| Method  | Intermediate Reward Advantage $\advseq_{\tau, \tt}$      | Single Trajectory Reward Advantage $\advseq_\tau$         |
|:---------|:--------------------------------------------|:--------------------------------------------------|
| REINFORCE | $\advseq_\tt = \rlret_\tt - b_\tt$                        | $\advseq = \rlret_{0,\tau} - b$                              |
| PPO     | $\advseq_\tt = \hat{\advseq}_\tt$ (GAE estimator) | GAE: the critic spreads the single reward over the steps |
| GRPO    | Relative advantages per candidate           | Same per-candidate advantage applied per step    |


And a comparison:

| Method | Gradient-Based | Main Objective | Key Characteristics | Stability & Sample Efficiency | Typical Application Domains |
|:--------|:----------------|:----------------|:---------------------|:------------------------------|:-----------------------------|
| **PPO** (Proximal Policy Optimization) | Yes | Maximize expected reward with clipped surrogate objective | Uses policy gradients with clipping to limit policy update magnitude, balancing exploration and exploitation | High stability; more sample efficient than vanilla policy gradients; widely used for continuous and discrete control tasks | Robotics, games, continuous control, benchmark RL tasks |
| **GRPO** (Group Relative Policy Optimization) | Yes | Optimize policy using group-relative advantage estimates | Does not require a separate value function; updates policy based on relative advantages within a group of candidate outputs | More memory efficient, stable; effective for large-scale policy optimization with reduced critic reliance | Training large language models, large-scale policy optimization |
| **REINFORCE** | Yes | Maximize expected cumulative reward by direct policy gradient | Uses Monte Carlo sampled returns, applies likelihood ratio trick; pure policy gradient without value function | High variance and sample inefficient; simpler but less stable than actor-critic or PPO | Fundamental policy gradient algorithm, baseline for many RL studies |


# REINFORCE (the "Original")

**Why this section?** The simplest method that uses an advantage; the reference point for PPO and GRPO.

REINFORCE is an early method (historically)
that is very close to the Policy Gradient Theorem formulation
- uses Advantage rather than the episode Return
- the Advantage subtracts a "baseline" from the episode Return

| Method  | Intermediate Reward Advantage $\advseq_{\tau, \tt}$      | Single Trajectory Reward Advantage $\advseq_\tau$         |
|:---------|:--------------------------------------------|:--------------------------------------------------|
| REINFORCE | $\advseq_\tt = \rlret_\tt - b_\tt$                        | $\advseq = \rlret_{0,\tau} - b$                              |
|

The purpose of subtracting a baseline is to
- reduce the magnitude of the gradients
- reduces the variance of the gradients
- and hence: the change in policy parameters $\theta$

In  single,  end-of-episode reward, the Advantage is typically computed as
$$
\advseq_{\tau,\tt} = \left( \rlret_{0,\tau} - b \right)
$$

where $b$ is a baseline
- computed as the Global Moving Average of episode returns
- over *all prompts*
- processed thus far (the baseline used for an episode is the value *before* updating it with that episode)


The Baseline Update Rule: 
- Every time you finish a rollout and get an episode return $\rlret_{0,\tau}$, 
- update the baseline:
$$b_{new} = (1 - w)\, b_{old} + w \, \rlret_{0,\tau}$$

where $w$ is a small weight of the moving average (not the learning rate $\alpha$)


The baseline calibrates the reward received for the current prompt/trajectory against all
previous prompts.

With intermediate rewards, the Advantage is typically computed as
$$
\advseq_{\tau,\tt} = \left( \rlret_{\tau, \tt} - b_\tt \right)
$$
- where $b_\tt$ is often a proxy for the *expected* Value function $\statevalfun_\pi(\stateseq_\tt)$ of state $\stateseq_\tt$
- if $b_\tt$ is a learned value function it needs a separate Critic (a running average does not)

It uses "Monte-Carlo sampling" method to estimate the gradient

**Note**

"Monte Carlo sampling" is a confusing way of saying: complete trajectory for the single prompt

The word "Monte Carlo" is thus used in the same sense as 
"Monte Carlo" in the [Temporal Difference methods we studied](RL_Value_based_model_free.ipynb#Monte-Carlo-(MC)-method)

## Pseudo code for REINFORCE

    # REINFORCE training for LLM
    for prompt in training_prompts:
        output = llm.generate(prompt)
        reward = evaluate_output(output) # Human or automated score
        logprob = llm.logprob(output, prompt)

        # Monte Carlo policy gradient update (no critic)
        baseline = compute_baseline() # Optional: running mean for variance reduction
        loss = -logprob * (reward - baseline)   # the baseline is a constant (no gradient)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


## Discussion

**Why this slide?** What are REINFORCE's weaknesses, in theory and in practice?

REINFORCE is very close to the vanilla Policy Gradient.

But it is considered high variance
- due to a source not related to the baseline


The Advantage depends on
$$\rlret_{\tau, \tt}$$

which is computed *based on the single trajectory*  $\tau$ that was encountered.

Thus, we are observing the Environment's Transition Probabilities 
- via a *single sample* from its distribution, for each step

The weight is therefore the return of a *single sampled episode*, which depends on the Environment's random responses: it fluctuates from episode to episode (high variance).

Subsequent methods will attempt to mitigate this source of variance.


    
In the *practical* implementation of REINFORCE
- we estimate $\rlret_{\tt}$
- over a *batch* of episodes $\tau_1, \ldots, \tau_G$ for the same prompt

so we have several episodes for the *same prompt*
- their mean return is a prompt-specific baseline
- and averaging the $G$ gradient terms lowers the variance

The "practical" version below uses a group baseline and is very similar to GRPO. The group mean includes the sample itself, which makes the baseline slightly dependent on the action; the leave-one-out mean avoids this.

    # Practical RL Loop Logic
    for prompt in batch:
        # 1. Sample G trajectories for the SAME prompt
        trajectories = model.generate(prompt, n=8)   # n = G, e.g. 8

        # 2. Get rewards
        rewards = reward_model(trajectories)

        # 3. Calculate Local Baseline (The "Modern" Way)
        # This cancels out prompt difficulty automatically
        baseline = rewards.mean() 

        # 4. Standardized Advantage
        advantages = (rewards - baseline) / (rewards.std() + 1e-8)

        # 5. Compute loss weighted by how much better/worse each sample was
        log_probs = model.logprobs(trajectories)   # one number per trajectory (summed over its tokens)
        loss = -log_probs * advantages

### Comparison: Theoretical vs. Practical REINFORCE

| Feature | Theoretical REINFORCE | Practical (e.g., GRPO / Batch RL) |
| :--- | :--- | :--- |
| **Rollouts per Prompt** | **1** | **Multiple (G = 4 to 64)** |
| **Baseline Type** | **Global Moving Average** (Historical) | **Local Group Mean** (Immediate) |
| **Variance** | **Very High** (Requires small learning rates) | **Reduced** (Brute-forced by sampling) |
| **Update Frequency** | Every single trajectory | After a group/batch of trajectories |
| **Compute Cost** | Low per update | High per update (heavy generation) |
| **Analogy** | Checking your lifetime golf average. | Comparing 10 swings from the same spot. |

# Surrogate Loss

**Why this section?** REINFORCE uses each batch of episodes once. To reuse a batch safely (PPO, next session) we need an objective we can optimize more than once.

The practical objective is a *surrogate*: a stand-in that is easier and safer to optimize
$$
\E_\tt \bigl[ r_\tt(\theta) \, \advseq_\tt \bigr], \qquad r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt \mid \stateseq_\tt)}{\pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)}
$$

- the episodes were generated by the *old* policy; the ratio $r_\tt$ re-weights them as if the new policy had generated them (an *importance weight*)
- the surrogate is a *local* model of the improvement over the old policy: at $\theta = \theta_{\text{old}}$ (ratio $1$) its gradient is the policy gradient with advantage $\advseq_\tt$
- far from $\theta_{\text{old}}$ the local model is no longer trustworthy, so we keep $\theta$ near $\theta_{\text{old}}$
- training minimizes the *negative* of the surrogate (a sign convention)


**Keeping $\theta$ near $\theta_{\text{old}}$**

| Mechanism | How | Used by |
|:--|:--|:--|
| trust region | a hard bound on the KL divergence between the old and the new policy | TRPO |
| clipping | the gradient of a sample vanishes once its ratio has left the range in the direction its advantage favours | PPO |
| KL penalty to a reference model | a penalty on drifting away from a frozen model | RLHF-style PPO, GRPO |


# What we established, and what it sets up

- **Established**
    - subtracting a baseline gives an advantage: the same gradient on average, less noise, and credit that can be positive or negative
    - REINFORCE weights by return minus a baseline
    - the surrogate loss is the practical objective: it keeps each update close to the old policy
- **Sets up**
    - next session: PPO (a clipped ratio, a critic, GAE) and GRPO (a group-relative advantage, no critic)
    - the self-study appendix collects the bias and variance of the estimators


# Appendix (self-study): bias and variance of the estimators

- these slides compare the estimators met in the value-based sessions and here: Monte Carlo, TD, dynamic programming, Q-learning
- they are not presented in class (tagged skip) but stay in the notebook


# Bias and Variance

We now have multiple methods for RL
- how can we compare them ?

We will introduce the concepts of Bias and Variance as one metric with which to compare methods.

To be concrete:

What are the advantages/disadvantages of Temporal Difference(1 step) vs Monte Carlo in updating
the Value function ?


Consider iteration $k$:

In TD:
- the value of the successor state is *not* the true end-of-episode $k$ return
- it is an *estimate* of the state's value based on the *previous* iteration
- updating one estimate using another estimated is called *bootstrapping*

In MC
- the value of the successor state is the *true* end-of-episode $k$ return

Because TD depends on an estimate
- we say that TD is *biased*
    - because it bootstraps from an estimate that is still inaccurate


Moreover, let's suppose that rewards are stochastic
- hence, each reward is a random variable

TD 1-step update 
- depends on **exactly one** random variable $$\hat\rew_{\tt+1}$$

TD k-step update
- depends on $k$ random variables: rewards for steps $\tt+1, \ldots, \tt+k$

MC update
- depends on **at least one** random variable
    - rewards of *all* subsequent states of the episode $\pi$
 $$\hat\rew_{\tt+1}, \ldots, $$

The variance of a sum of $k$ *independent* random variables is the sum of their variances
- *exactly* $k$ times the variance of each, if the variances are equal

So the variance of the $\statevalfun$ estimated by the various methods
- is smallest for TD 1-step
- increasing with $k$ for TD k-step
- is largest for Monte Carlo


**But**
- the *bias* decreases in the opposite direction of the increase of the variance
- estimate based on more information

Thus
- TD(0) is
    - low variance (one random variable), high bias (based on estimates)
- MC is
    - high variance (*at least* one random variable), **zero** bias (true values, not estimates)
    

## Bias and Variance of the methods presented

| Method                   | Bias          | Variance      | Notes                                                                                                        |
|:------------------------ |:------------- |:------------- |:------------------------------------------------------------------------------------------------------------ |
| DP (Dynamic Programming) | None (exact expectation) | None | Uses the full environment model; updates use true expectations; the values still have error until converged. |
| MC (Monte Carlo)         | None          | High          | Unbiased (targets equal expected return), but episodes can have widely varying returns.                      |
| TD (Temporal Difference) | Moderate      | Moderate/Low  | Bootstrap introduces bias (approximate next value), but reduces variance compared to MC.                     |
| V-learning (State Value) | Moderate      | Low           | Like TD if bootstrapped; low-variance due to value averaging.                                                |
| Q-learning               | Moderate/High | Moderate/High | Off-policy; bias from bootstrapping and from the max over noisy estimates (overestimation); variance from sampling.                         |

**Key Points**

DP (Dynamic Programming): Lowest bias and variance, but needs a full model (rarely available in practice).

MC: Unbiased estimates, but high variance due to sampling full returns.

TD: Bootstrapping introduces bias but greatly lowers variance versus MC.

V-learning: Similar profile to TD (if bootstrapped); often lower variance than Q-learning due to value averaging.

Q-learning: Can be more biased/variable due to off-policy backups and maximization over sampled estimates.



# Sources of Variance in RL

We introduced Variance as a way of comparing TD and MC for value-based methods.

But variance is present in many RL methods and is considered 
 a potential impediment to Learning via RL.
- bigger updates in
    - Value function for Value-based methods
    - Parameters for Policy-based methods
- can cause large changes in Policy
- which can lead to unstable training

We list the types of methods affected by each cause.

There multiple sources of Variance, which we summarize below
- stochastic environment
    - stochastic rewards and state transitions
- stochastic policy
- sparse rewards
    - per-episode vs per-step rewards
    - reward estimates become more noisy
- bootstrapping    


| Source of Variance             | Description                                                       | Methods Most Affected                |
|:-------------------------------|:-------------------------------------------------------------------|:--------------------------------------|
| Environmental Stochasticity   | Random rewards and transitions cause unpredictable outcomes        | Monte Carlo, TD, Policy Gradients    |
| Policy Stochasticity          | Probabilistic (non-deterministic) action selection                | MC, On-policy, Policy Gradients      |
| Long Trajectory Aggregation   | Returns summed over many steps compound the randomness             | Monte Carlo, n-step methods          |
| Sparse or Delayed Rewards     | Few positive signals lead to noisy estimates                       | Monte Carlo, Value-based             |
| Bootstrapping Error           | Using own predictions as targets introduces bias, but lowers variance | TD, Q-Learning                    |
| Credit Assignment Difficulty  | Uncertainty in linking actions to future rewards increases variance | Policy Gradients, MC                 |
| Sample Size & Exploration     | Insufficient samples or aggressive exploration cause wide fluctuations| All methods                       |
| Model/Optimization Instability| Neural network or optimizer issues amplify variance                | Deep RL, Policy Gradient methods     |
| Non-stationarity              | Changing environment or policy during training                     | All methods, esp. online learning    |

**Key Points:**  
- Variance comes from randomness in data, policy, training procedure, and modeling choices.  
- Some sources can be controlled with design choices (e.g., baselines, bootstrapping, averaging), while others are intrinsic to RL problems.

As we introduce new RL methods
- one motivation is to reduce variance

Some potential ways to reduce variance
- $n \gt 1$-step ahead Temporal Difference
- estimating updates in mini-batches
    - as in Gradient Descent


In [2]:
print("Done")

Done
